# Pipeline de Big Data — Predicción de Incendios Forestales
## Exploración y Exportación del Dataset
**Alumno:** Victor Andres Aguila Lopez  
**Materia:** Big Data | UPIIT-IPN   
**Dataset:** 1.88 Million US Wildfires (Kaggle, CC0-1.0)

En este notebook se realiza la exploración inicial del archivo SQLite descargado desde Kaggle,
se identifican las tablas disponibles, se analiza el esquema de columnas de la tabla principal
`Fires`, y se exportan las columnas relevantes a formato CSV para su posterior ingesta en HDFS.

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('FPA_FOD_20170508.sqlite')

# Ver todas las tablas disponibles
tablas = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(tablas)

                                  name
0                      spatial_ref_sys
1                   spatialite_history
2                      sqlite_sequence
3                     geometry_columns
4                  spatial_ref_sys_aux
5               views_geometry_columns
6               virts_geometry_columns
7          geometry_columns_statistics
8    views_geometry_columns_statistics
9    virts_geometry_columns_statistics
10        geometry_columns_field_infos
11  views_geometry_columns_field_infos
12  virts_geometry_columns_field_infos
13               geometry_columns_time
14               geometry_columns_auth
15         views_geometry_columns_auth
16         virts_geometry_columns_auth
17                  sql_statements_log
18                        SpatialIndex
19                ElementaryGeometries
20                                 KNN
21                               Fires
22                     idx_Fires_Shape
23                idx_Fires_Shape_node
24               idx_Fire

### Tablas disponibles en el archivo SQLite
El archivo `FPA_FOD_20170508.sqlite` contiene múltiples tablas de metadatos geoespaciales
propias del formato SpatiaLite. La tabla de interés para este proyecto es **`Fires`** (índice 21),
que contiene los registros históricos de incendios forestales.
A continuación se inspecciona su esquema completo.

In [2]:
# Ver el esquema completo de la tabla Fires
schema = pd.read_sql_query("PRAGMA table_info(Fires);", conn)
print(schema[['name', 'type']].to_string())

                          name       type
0                     OBJECTID    INTEGER
1                       FOD_ID      int32
2                       FPA_ID  text(100)
3           SOURCE_SYSTEM_TYPE  text(255)
4                SOURCE_SYSTEM   text(30)
5        NWCG_REPORTING_AGENCY  text(255)
6       NWCG_REPORTING_UNIT_ID  text(255)
7     NWCG_REPORTING_UNIT_NAME  text(255)
8        SOURCE_REPORTING_UNIT   text(30)
9   SOURCE_REPORTING_UNIT_NAME  text(255)
10        LOCAL_FIRE_REPORT_ID  text(255)
11           LOCAL_INCIDENT_ID  text(255)
12                   FIRE_CODE   text(10)
13                   FIRE_NAME  text(255)
14     ICS_209_INCIDENT_NUMBER  text(255)
15                ICS_209_NAME  text(255)
16                     MTBS_ID  text(255)
17              MTBS_FIRE_NAME   text(50)
18                COMPLEX_NAME  text(255)
19                   FIRE_YEAR      int16
20              DISCOVERY_DATE   realdate
21               DISCOVERY_DOY      int32
22              DISCOVERY_TIME    

### Esquema completo de la tabla Fires
La tabla `Fires` contiene **38 columnas** con información geoespacial, temporal, causal y de magnitud.
Para este proyecto se seleccionarán **12 columnas relevantes**:
`FIRE_YEAR`, `DISCOVERY_DATE`, `DISCOVERY_DOY`, `STAT_CAUSE_DESCR`, `FIRE_SIZE`,
`FIRE_SIZE_CLASS`, `LATITUDE`, `LONGITUDE`, `STATE`, `CONT_DATE`, `CONT_DOY`, `OWNER_DESCR`.

In [4]:
# Confirmar número total de filas
total = pd.read_sql_query("SELECT COUNT(*) as total FROM Fires;", conn)
print(total)

     total
0  1880465


### Resultado: 1,880,465 registros confirmados
El dataset contiene casi **1.88 millones de registros** históricos de incendios forestales
en EE.UU. entre 1992 y 2015, lo que justifica el uso del ecosistema Hadoop para su procesamiento.

### Extracción de columnas relevantes
Se construye una query SQL para extraer únicamente las 12 columnas seleccionadas,
reduciendo el volumen de datos a procesar en las etapas posteriores del pipeline.

In [5]:
query = """
SELECT FIRE_YEAR, DISCOVERY_DATE, DISCOVERY_DOY, STAT_CAUSE_DESCR,
       FIRE_SIZE, FIRE_SIZE_CLASS, LATITUDE, LONGITUDE, STATE,
       CONT_DATE, CONT_DOY, OWNER_DESCR
FROM Fires
"""

df = pd.read_sql_query(query, conn)
print(f"Filas exportadas: {len(df)}")
print(df.head())
print(df.dtypes)

Filas exportadas: 1880465
   FIRE_YEAR  DISCOVERY_DATE  DISCOVERY_DOY STAT_CAUSE_DESCR  FIRE_SIZE  \
0       2005       2453403.5             33    Miscellaneous       0.10   
1       2004       2453137.5            133        Lightning       0.25   
2       2004       2453156.5            152   Debris Burning       0.10   
3       2004       2453184.5            180        Lightning       0.10   
4       2004       2453184.5            180        Lightning       0.10   

  FIRE_SIZE_CLASS   LATITUDE   LONGITUDE STATE  CONT_DATE  CONT_DOY  \
0               A  40.036944 -121.005833    CA  2453403.5      33.0   
1               A  38.933056 -120.404444    CA  2453137.5     133.0   
2               A  38.984167 -120.735556    CA  2453156.5     152.0   
3               A  38.559167 -119.913333    CA  2453189.5     185.0   
4               A  38.559167 -119.933056    CA  2453189.5     185.0   

        OWNER_DESCR  
0              USFS  
1              USFS  
2  STATE OR PRIVATE  
3       

### Vista previa del dataset exportado
Las primeras filas confirman la estructura correcta del dataframe:
- `FIRE_SIZE_CLASS = A` indica incendios de menos de 0.25 acres (los más frecuentes)
- `DISCOVERY_DATE` aparece en formato Julian Date (número de días desde el 1 de enero de 4713 a.C.)
- Los primeros registros corresponden a California (CA), estado con más incendios en el dataset

### Análisis de calidad de datos — Valores nulos
Se verifica la integridad de cada columna antes de exportar al CSV definitivo.

In [6]:
# Revisar valores nulos antes de exportar
print(df.isnull().sum())

FIRE_YEAR                0
DISCOVERY_DATE           0
DISCOVERY_DOY            0
STAT_CAUSE_DESCR         0
FIRE_SIZE                0
FIRE_SIZE_CLASS          0
LATITUDE                 0
LONGITUDE                0
STATE                    0
CONT_DATE           891531
CONT_DOY            891531
OWNER_DESCR              0
dtype: int64


### Resultado del análisis de nulos
| Columna | Nulos | Porcentaje |
|---|---|---|
| CONT_DATE | 891,531 | 47.4% |
| CONT_DOY | 891,531 | 47.4% |
| Resto de columnas | 0 | 0% |

Los valores nulos en `CONT_DATE` y `CONT_DOY` corresponden a incendios para los cuales
no se registró formalmente su fecha de contención — situación común en el dataset.
El resto de las columnas seleccionadas no presenta valores nulos, garantizando
integridad en las variables usadas para el modelo predictivo.

### Exportación a CSV
Se exportan los 1,880,465 registros con las 12 columnas seleccionadas a formato CSV,
listo para ser ingestado en HDFS.

In [7]:
# Exportar a CSV
df.to_csv('wildfires.csv', index=False)
conn.close()
print("CSV exportado correctamente")

CSV exportado correctamente


### CSV exportado correctamente
El archivo `wildfires.csv` fue generado exitosamente en la carpeta del proyecto.
A continuación se verifica su tamaño en disco.

In [8]:
import os
print(f"Tamaño: {os.path.getsize('wildfires.csv') / (1024*1024):.2f} MB")

Tamaño: 153.12 MB


### Resultado final
El archivo `wildfires.csv` ocupa **153.12 MB** en disco con 1,880,465 registros y 12 columnas.

**Siguiente paso:** subir el archivo a HDFS para su procesamiento distribuido:
```bash
hdfs dfs -put wildfires.csv /user/vaal/bigdata_incendios/raw/
```
El pipeline continúa en las fases de análisis exploratorio con Apache Hive y Apache Pig.